In [253]:
# Parameters
from pathlib import Path

REPO_ROOT = Path("..").resolve()
if not (REPO_ROOT / "algorithms").is_dir():
    REPO_ROOT = Path.cwd().resolve()

OUTPUT_DIR = REPO_ROOT / "plotting" / "coordination"
USE_CACHED_VALUES = True
ROLLOUTS_PER_COUNT = 8
COUNTS = None 
OVERWRITE_OUTPUT_DIR = False
DPI = 300
FONT_FAMILY = "Charter"
CMAP = "RdPu"

TICK_FONTSIZE = 15
AXIS_LABEL_FONTSIZE = 20
CMAP_TITLE_FONTSIZE = 18

TASKS = [
    {
        "task_key": "spread",
        "task_id": "simple_spread_dynamic_hard",
        "task_label": "Spread",
        "results_dir": REPO_ROOT / "results" / "spread",
        "algorithm": "pimac_v6",
        "config_name": "active_03",
    },
    {
        "task_key": "lbf_hard",
        "task_id": "lbf_hard",
        "task_label": "LBF",
        "results_dir": REPO_ROOT / "results" / "lbf_hard",
        "algorithm": "pimac_v6",
        "config_name": "active_01",
    },
    {
        "task_key": "rware",
        "task_id": "robotic_warehouse_dynamic",
        "task_label": "RWARE",
        "results_dir": REPO_ROOT / "results" / "rware",
        "algorithm": "pimac_v6",
        "config_name": "active_01",
    },
]


In [254]:
import csv
import importlib
import json
import shutil
import sys
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import torch

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from algorithms.registry import get_algorithm_class


In [255]:
def load_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with Path(path).open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, sort_keys=True)


def read_csv(path):
    with Path(path).open("r", newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def write_csv(path, rows):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def checkpoint_path_for_run(run_dir):
    final_checkpoint = run_dir / "final_checkpoint.pt"
    if final_checkpoint.is_file():
        return final_checkpoint
    raise FileNotFoundError(f"Missing checkpoint in {run_dir}")


def discover_runs(task):
    algorithm_dir = Path(task["results_dir"]) / task["algorithm"]

    runs = []
    for config_snapshot_path in sorted(algorithm_dir.glob("*/config_snapshot.json")):
        run_dir = config_snapshot_path.parent
        summary_path = run_dir / "summary.json"
        if not summary_path.is_file():
            continue

        config_snapshot = load_json(config_snapshot_path)
        if str(config_snapshot.get("algorithm")) != task["algorithm"]:
            continue
        task_config = dict(config_snapshot.get("task_config", {}))
        if str(task_config.get("task_name")) != task["task_id"]:
            continue
        config_name = Path(str(config_snapshot.get("algorithm_config_path", ""))).stem
        if config_name != task["config_name"]:
            continue

        summary = load_json(summary_path)
        runs.append(
            {
                "seed": int(summary["seed"]),
                "run_dir": run_dir,
                "checkpoint_path": checkpoint_path_for_run(run_dir),
                "config_snapshot": config_snapshot,
            }
        )
    if not runs:
        raise RuntimeError(f"{task['task_label']} {task['algorithm']} {task['config_name']}")
    return runs


def task_script_module(task_id):
    return importlib.import_module(f"{task_id}.run")


def load_policy(task, run):
    task_config = dict(run["config_snapshot"]["task_config"])
    task_script = task_script_module(task["task_id"])
    env_spec = task_script.build_env_spec(task_config, run["seed"])
    learner_cls = get_algorithm_class(task["algorithm"])
    learner = learner_cls.load_checkpoint(
        run["checkpoint_path"],
        env_spec=env_spec,
        config=dict(run["config_snapshot"]["algorithm_config"]),
        device="cpu",
    )
    learner.set_eval_mode()
    return learner, task_config, task_script, env_spec


def analysis_counts(task_config):
    if COUNTS is not None:
        return [int(value) for value in COUNTS]
    if str(task_config.get("task_type", "")).startswith("dynamic_team"):
        return [int(value) for value in task_config["eval_counts"]]
    return [int(task_config["n_agents"])]


def transform_obs(task_script, env_spec, obs):
    return task_script.prepare_observation(obs, env_spec)


def extract_teacher_context(critic_outputs):
    return critic_outputs[2]


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=np.float32).reshape(-1)
    b = np.asarray(b, dtype=np.float32).reshape(-1)
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    return 0.0 if denom == 0.0 else float(np.dot(a, b) / denom)


In [ ]:
def summarize_seed_count_rows(step_rows):
    grouped = defaultdict(list)
    for row in step_rows:
        grouped[(row["run_label"], int(row["n_agents"]))].append(row)

    summary_rows = []
    for (run_label, n_agents), rows in sorted(grouped.items(), key=lambda item: (str(item[0][0]), int(item[0][1]))):
        first = rows[0]
        gate_values = [float(row["gate"]) for row in rows if row.get("gate") is not None]
        summary_rows.append(
            {
                "task": first["task"],
                "task_label": first["task_label"],
                "algorithm": first["algorithm"],
                "rank": int(first["rank"]),
                "trial_number": int(first["trial_number"]),
                "config_name": first["config_name"],
                "run_label": run_label,
                "n_agents": int(n_agents),
                "samples": int(len(rows)),
                "teacher_alignment_cosine_mean": float(np.mean([float(row["teacher_alignment_cosine"]) for row in rows])),
                "teacher_alignment_mse_mean": float(np.mean([float(row["teacher_alignment_mse"]) for row in rows])),
                "ctx_signal_mean": float(np.mean([float(row["ctx_signal_mean"]) for row in rows])),
                "gate_mean": float(np.mean(gate_values)) if gate_values else None,
                "gate_std": float(np.std(gate_values)) if gate_values else None,
                "teacher_ctx_norm_mean": float(np.mean([float(row["teacher_ctx_norm"]) for row in rows])),
                "student_ctx_norm_mean": float(np.mean([float(row["student_ctx_norm"]) for row in rows])),
            }
        )
    return summary_rows


def collect_task_trace(task):
    runs = discover_runs(task)
    first_task_config = dict(runs[0]["config_snapshot"]["task_config"])
    dynamic_task = str(first_task_config.get("task_type", "")).startswith("dynamic_team")
    counts = analysis_counts(first_task_config)
    step_rows = []

    for run in runs:
        run_label = f"s{int(run['seed'])}"
        learner, task_config, task_script, env_spec = load_policy(task, run)
        for n_agents in counts:
            print(f"[{task['task_label']}] {task['algorithm']} {task['config_name']} {run_label} n={n_agents}")
            for episode_index in range(int(ROLLOUTS_PER_COUNT)):
                rollout_seed = int(run["seed"] + (1000 * int(n_agents)) + episode_index)
                torch.manual_seed(rollout_seed)
                env = task_script.make_env(task_config, seed=rollout_seed, n_agents=int(n_agents), render_mode=None)

                obs, _ = env.reset(seed=rollout_seed)
                learner.reset_episode()
                step_index = 0
                try:
                    while obs:
                        agent_ids = sorted(obs.keys(), key=str)
                        if not agent_ids:
                            break

                        actions = {}
                        row_buffer = []
                        student_contexts = []

                        with torch.no_grad():
                            for agent_position, agent_id in enumerate(agent_ids):
                                obs_value = transform_obs(task_script, env_spec, obs[agent_id])
                                obs_tensor = torch.as_tensor(obs_value, dtype=torch.float32, device=learner.device).view(1, 1, -1)
                                hidden_dim = learner.actor_net.rnn.hidden_size
                                hidden_state = learner._get_hidden_state(agent_id, hidden_dim)
                                logits, next_hidden_state, aux = learner.actor_net(obs_tensor, hidden_state, return_aux=True)
                                learner._set_hidden_state(agent_id, next_hidden_state)
                                action = int(torch.distributions.Categorical(logits=logits.squeeze(0).squeeze(0)).sample().item())
                                actions[str(agent_id)] = action

                                student_ctx = aux["ctx_mu"].squeeze(0).squeeze(0).detach().cpu().numpy()
                                signal_key = "ctx_reliance" if "ctx_reliance" in aux else ("ctx_log_uncertainty" if "ctx_log_uncertainty" in aux else "ctx_logvar")
                                ctx_signal = aux[signal_key].squeeze(0).squeeze(0).detach().cpu().numpy()
                                row_buffer.append(
                                    {
                                        "task": task["task_id"],
                                        "task_label": task["task_label"],
                                        "algorithm": task["algorithm"],
                                        "rank": 0,
                                        "trial_number": int(run["seed"]),
                                        "config_name": task["config_name"],
                                        "run_label": run_label,
                                        "episode_index": int(episode_index),
                                        "step_index": int(step_index),
                                        "seed": int(rollout_seed),
                                        "n_agents": int(n_agents),
                                        "agent_id": str(agent_id),
                                        "agent_position": int(agent_position),
                                        "action": int(action),
                                        "ctx_signal_mean": float(np.mean(ctx_signal)),
                                        "student_ctx_norm": float(np.linalg.norm(student_ctx)),
                                        "gate": float(aux["gate"].squeeze(0).squeeze(0).detach().cpu().item()) if "gate" in aux else None,
                                    }
                                )
                                student_contexts.append(np.asarray(student_ctx, dtype=np.float32))

                            stacked_obs = np.stack([transform_obs(task_script, env_spec, obs[agent_id]) for agent_id in agent_ids], axis=0)
                            obs_tensor = torch.as_tensor(stacked_obs, dtype=torch.float32, device=learner.device).view(1, 1, len(agent_ids), -1)
                            active_mask = torch.ones(1, 1, len(agent_ids), dtype=torch.float32, device=learner.device)
                            teacher_context = extract_teacher_context(learner.critic(obs_tensor, active_mask, return_details=True))
                            teacher_contexts = teacher_context.squeeze(0).squeeze(0).detach().cpu().numpy()

                        for row, student_ctx, teacher_ctx in zip(row_buffer, student_contexts, teacher_contexts):
                            row["teacher_alignment_cosine"] = cosine_similarity(student_ctx, teacher_ctx)
                            row["teacher_alignment_mse"] = float(np.mean((student_ctx - teacher_ctx.reshape(-1)) ** 2))
                            row["teacher_ctx_norm"] = float(np.linalg.norm(teacher_ctx))
                            step_rows.append(row)

                        obs, _, terminations, truncations, _ = env.step(actions)
                        done = {
                            agent_id: bool(terminations.get(agent_id, False) or truncations.get(agent_id, False))
                            for agent_id in set(terminations) | set(truncations)
                        }
                        obs = {
                            str(agent_id): np.asarray(value, dtype=np.float32)
                            for agent_id, value in obs.items()
                            if not done.get(agent_id, False)
                        }
                        step_index += 1
                finally:
                    env.close()

    seed_count_rows = summarize_seed_count_rows(step_rows)
    return {"task": task, "counts": counts, "seeds": [int(run["seed"]) for run in runs], "seed_count_rows": seed_count_rows}


def build_heatmap_rows(traces):
    heatmap_rows = []
    for trace in traces:
        grouped = defaultdict(list)
        for row in trace["seed_count_rows"]:
            grouped[int(row["n_agents"])].append(row)
        for n_agents, rows in sorted(grouped.items()):
            alignment_values = [float(row["teacher_alignment_cosine_mean"]) for row in rows]
            gate_values = [float(row["gate_mean"]) for row in rows if row.get("gate_mean") is not None]
            heatmap_rows.append(
                {
                    "task": trace["task"]["task_id"],
                    "task_key": trace["task"]["task_key"],
                    "task_label": trace["task"]["task_label"],
                    "algorithm": trace["task"]["algorithm"],
                    "config_name": trace["task"]["config_name"],
                    "n_agents": int(n_agents),
                    "model_count": int(len(rows)),
                    "alignment_mean": float(np.mean(alignment_values)),
                    "alignment_std": float(np.std(alignment_values, ddof=1)) if len(alignment_values) > 1 else 0.0,
                    "gate_mean": float(np.mean(gate_values)) if gate_values else None,
                    "gate_std": float(np.std(gate_values, ddof=1)) if len(gate_values) > 1 else 0.0,
                }
            )
    return heatmap_rows


In [257]:
def task_output_name(task):
    return task["task_label"].lower().replace(" ", "_")


def plot_alignment_gate_heatmap(rows, output_path, title=None):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    task_labels = list(dict.fromkeys(str(row["task_label"]) for row in rows))
    counts = sorted({int(row["n_agents"]) for row in rows})
    alignment_mean = np.full((len(task_labels), len(counts)), np.nan, dtype=np.float32)
    gate_mean = np.full_like(alignment_mean, np.nan)
    gate_std = np.full_like(alignment_mean, np.nan)

    for row in rows:
        task_index = task_labels.index(str(row["task_label"]))
        count_index = counts.index(int(row["n_agents"]))
        alignment_mean[task_index, count_index] = float(row["alignment_mean"])
        gate_mean[task_index, count_index] = float(row["gate_mean"])
        gate_std[task_index, count_index] = float(row["gate_std"])

    with plt.rc_context({"font.family": "serif", "font.serif": [FONT_FAMILY, "XCharter", "Bitstream Charter", "DejaVu Serif"]}):
        fig_height = max(2.2, 0.62 * len(task_labels) + 1.55)
        fig, axis = plt.subplots(figsize=(1.18 * len(counts) + 2.6, fig_height))
        image = axis.imshow(np.ma.masked_invalid(alignment_mean), aspect="auto", cmap=CMAP, vmin=0.0, vmax=1.0)
        axis.set_xticks(np.arange(len(counts)))
        axis.set_xticklabels([str(value) for value in counts], fontsize=TICK_FONTSIZE)
        axis.set_yticks([])
        axis.set_xlabel("Number of agents", fontsize=AXIS_LABEL_FONTSIZE)

        for row_index in range(alignment_mean.shape[0]):
            for col_index in range(alignment_mean.shape[1]):
                # if not np.isfinite(alignment_mean[row_index, col_index]):
                #     continue
                cell_text = "NA" if not np.isfinite(gate_mean[row_index, col_index]) else f"{gate_mean[row_index, col_index]:.2f}\n±{gate_std[row_index, col_index]:.2f}"
                text_color = "white" #if alignment_mean[row_index, col_index] < 0.55 else "#111111"
                axis.text(col_index, row_index, cell_text, ha="center", va="center", color=text_color, fontsize=TICK_FONTSIZE)

        cbar_ax = fig.add_axes([1.0, 0.05, 0.02, 1])
        colorbar = fig.colorbar(image, ax=axis, fraction=0.03, pad=0.02, cmap=CMAP, cax=cbar_ax)
        colorbar.set_label("Cosine alignment", fontsize=CMAP_TITLE_FONTSIZE, labelpad=3)
        colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)
        fig.tight_layout()
        fig.savefig(output_path, dpi=int(DPI), bbox_inches="tight")
        plt.close(fig)
    return output_path

In [258]:
if OUTPUT_DIR.exists() and OVERWRITE_OUTPUT_DIR and not USE_CACHED_VALUES:
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plot_inputs = []

if USE_CACHED_VALUES:
    try:
        all_heatmap_rows = read_csv(OUTPUT_DIR / "alignment_heatmap_rows.csv")
        for task in TASKS:
            task_dir = OUTPUT_DIR / task_output_name(task)
            task_heatmap_rows = read_csv(task_dir / "alignment_heatmap_rows.csv")
            summary_path = task_dir / "summary_by_seed_count.csv"
            summary_rows = read_csv(summary_path) if summary_path.is_file() else []
            plot_inputs.append(
                {
                    "task": task,
                    "task_dir": task_dir,
                    "heatmap_rows": task_heatmap_rows,
                    "counts": sorted({int(row["n_agents"]) for row in task_heatmap_rows}),
                    "seeds": sorted({int(row["trial_number"]) for row in summary_rows}) if summary_rows else [],
                }
            )
    except:
        print("no cached values")
        USE_CACHED_VALUES = False
if not USE_CACHED_VALUES:
    traces = [collect_task_trace(task) for task in TASKS]
    all_heatmap_rows = build_heatmap_rows(traces)
    write_csv(OUTPUT_DIR / "alignment_heatmap_rows.csv", all_heatmap_rows)
    for trace in traces:
        task = trace["task"]
        task_dir = OUTPUT_DIR / task_output_name(task)
        task_dir.mkdir(parents=True, exist_ok=True)
        task_heatmap_rows = build_heatmap_rows([trace])
        write_csv(task_dir / "summary_by_seed_count.csv", trace["seed_count_rows"])
        write_csv(task_dir / "alignment_heatmap_rows.csv", task_heatmap_rows)
        plot_inputs.append(
            {
                "task": task,
                "task_dir": task_dir,
                "heatmap_rows": task_heatmap_rows,
                "counts": [int(value) for value in trace["counts"]],
                "seeds": [int(value) for value in trace["seeds"]],
            }
        )

plot_manifest = {"tasks": {}}

for item in plot_inputs:
    task = item["task"]
    task_dir = item["task_dir"]
    task_heatmap_rows = item["heatmap_rows"]
    title = "Cosine alignment"
    heatmap_path = plot_alignment_gate_heatmap(task_heatmap_rows, task_dir / f"{task["task_key"]}_heatmap.png", title=title)
    print(heatmap_path)


no cached values


/Users/akman/pimac/venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


[Spread] pimac_v6 active_03 s42 n=1
[Spread] pimac_v6 active_03 s42 n=2
[Spread] pimac_v6 active_03 s42 n=3
[Spread] pimac_v6 active_03 s42 n=4
[Spread] pimac_v6 active_03 s42 n=5
[Spread] pimac_v6 active_03 s42 n=6
[Spread] pimac_v6 active_03 s42 n=7
[Spread] pimac_v6 active_03 s42 n=8
[Spread] pimac_v6 active_03 s42 n=9
[Spread] pimac_v6 active_03 s42 n=10
[Spread] pimac_v6 active_03 s43 n=1
[Spread] pimac_v6 active_03 s43 n=2
[Spread] pimac_v6 active_03 s43 n=3
[Spread] pimac_v6 active_03 s43 n=4
[Spread] pimac_v6 active_03 s43 n=5
[Spread] pimac_v6 active_03 s43 n=6
[Spread] pimac_v6 active_03 s43 n=7
[Spread] pimac_v6 active_03 s43 n=8
[Spread] pimac_v6 active_03 s43 n=9
[Spread] pimac_v6 active_03 s43 n=10
[Spread] pimac_v6 active_03 s44 n=1
[Spread] pimac_v6 active_03 s44 n=2
[Spread] pimac_v6 active_03 s44 n=3
[Spread] pimac_v6 active_03 s44 n=4
[Spread] pimac_v6 active_03 s44 n=5
[Spread] pimac_v6 active_03 s44 n=6
[Spread] pimac_v6 active_03 s44 n=7
[Spread] pimac_v6 active_0

/var/folders/p5/mttkf5jd5_v5v48m7kzcwp840000gn/T/ipykernel_47457/3113848823.py:43: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/p5/mttkf5jd5_v5v48m7kzcwp840000gn/T/ipykernel_47457/3113848823.py:43: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


/Users/akman/pimac/plotting/coordination/spread/spread_heatmap.png
/Users/akman/pimac/plotting/coordination/lbf/lbf_hard_heatmap.png
/Users/akman/pimac/plotting/coordination/rware/rware_heatmap.png


/var/folders/p5/mttkf5jd5_v5v48m7kzcwp840000gn/T/ipykernel_47457/3113848823.py:43: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
